# Building the AMR Training Dataset — Corrected (Leakage-Audited) Version

Rebuilds **`AMR_Training_Dataset_Fixed.csv`** from the raw WHO GLASS exports in `AMR_Data/`.

This version repairs four benchmark defects found in the legacy
`AMR_Training_Dataset_Final.csv`:

| Defect | Legacy behaviour | Correction |
|---|---|---|
| D1 target replication | rows with both BLOOD and Bloodstream records had `lag1` set to their own current value | lags are a pure same-series shift |
| D2 stratum crossing | lags grouped by country/pathogen/antibiotic but **not** specimen, chaining values across specimens | group by all four keys |
| D3 post-hoc flags | `Country_Observation_Count` summed the full 2020–2023 period | count only years strictly before the row's year |
| D4 full-panel imputation | missing DID filled with the global median over all years | median of observed 2020–2021 consumption |

The legacy file also dropped 367 genuine observations; the corrected extraction keeps them
(6,276 rows vs 5,909). The legacy CSV is retained in this repository unchanged, because the
paper replays the defective pipeline on it as a regression test.

## 1. Setup

In [1]:
import os
import csv
import io
import re
from pathlib import Path

import numpy as np
import pandas as pd

BASE     = Path('AMR_Data')
RES_DIR  = BASE / 'Resistance to antibiotics in the selected calendar year'
RES_2020 = BASE / '2020'
DID_DIR  = BASE / 'Use by antimicrobial classes'
XLSX     = Path('income_group_world_bank.xlsx')

TARGET_YEARS = [2021, 2022, 2023]   # rows in the final dataset
HQ_THRESHOLD = 176                  # high-quality surveillance flag threshold (time-causal count)

PAPER_COUNTRIES = [
    'AUT','BEL','BFA','BGD','BHR','BRN','CHE','CZE','DEU','DNK',
    'EGY','ETH','FIN','FRA','GBR','GEO','HRV','IDN','IRL','IRN',
    'IRQ','ITA','JOR','KOR','KWT','LAO','LTU','LVA','MLT','NOR',
    'NPL','OMN','PAK','POL','QAT','RUS','SAU','SGP','TUN','TZA',
    'UGA','UKR','ZAF','ZMB'
]
print(f'{len(PAPER_COUNTRIES)} countries retained')

44 countries retained


## 2. Parser for the raw GLASS exports
Every raw WHO CSV stores the per-country rows under a `"Data for boxplots"` section with the
header signature `"Specimen","PathogenName","AntibioticName","Iso3",...`.

In [2]:
def extract_boxplots(path):
    text = open(path, encoding='utf-8-sig', errors='replace').read()
    lines = text.splitlines()
    out = []
    for i, ln in enumerate(lines):
        if ln.startswith('"Specimen","PathogenName","AntibioticName","Iso3"'):
            header = next(csv.reader(io.StringIO(ln)))
            for dln in lines[i + 1:]:
                s = dln.strip()
                if not s.startswith('"') or s.startswith('"Specimen"'):
                    break
                parts = next(csv.reader(io.StringIO(s)))
                if len(parts) < len(header):
                    break
                out.append(dict(zip(header, parts)))
            break
    return out

def get_year(fn):
    m = re.search(r'(20[0-2][0-9])', fn)
    return int(m.group(1)) if m else None

def extract_did(folder):
    out = []
    for fn in sorted(os.listdir(folder)):
        if 'distribution' not in fn or not fn.endswith('.csv'):
            continue
        text = open(folder / fn, encoding='utf-8-sig', errors='replace').read()
        myr = re.search(r'Year filter:\s*(\d{4})', text)
        mcls = re.search(r'Antimicrobial class filter:\s*([^"\n]+)', text)
        if not myr:
            continue
        yr = int(myr.group(1))
        cls = mcls.group(1).strip() if mcls else '?'
        ls = text.splitlines()
        for i, ln in enumerate(ls):
            if ln.startswith('"CountryTerritoryArea","COUNTRY","WHORegionName","footnotes","DID'):
                h = next(csv.reader(io.StringIO(ln)))
                for dl in ls[i + 1:]:
                    s = dl.strip()
                    if not s.startswith('"'):
                        break
                    pa = next(csv.reader(io.StringIO(s)))
                    if len(pa) < len(h):
                        continue
                    d = dict(zip(h, pa))
                    out.append({'Year': yr, 'Class': cls, 'Iso3': d['COUNTRY'],
                                'DID': pd.to_numeric(d.get('DID_antibiotics'), errors='coerce')})
                break
    return out

print('parsers defined')

parsers defined


## 3. Resistance rows (2020–2023) and antibiotic classes

In [3]:
ABX_CLASS = {
    'Amikacin': 'Other', 'Gentamicin': 'Other', 'Minocycline': 'Other',
    'Tigecycline': 'Other', 'Oxacillin': 'Other', 'Azithromycin': 'Other',
    'Co-trimoxazole': 'Sulfonamides',
    'Penicillin G': 'Penicillins', 'Ampicillin': 'Penicillins',
    'Cefepime': 'Cephalosporins', 'Cefotaxime': 'Cephalosporins',
    'Ceftazidime': 'Cephalosporins', 'Ceftriaxone': 'Cephalosporins', 'Cefixime': 'Cephalosporins',
    'Ciprofloxacin': 'Fluoroquinolones', 'Levofloxacin': 'Fluoroquinolones',
    'Imipenem': 'Carbapenems', 'Meropenem': 'Carbapenems',
    'Ertapenem': 'Carbapenems', 'Doripenem': 'Carbapenems',
    'Colistin': 'Polymyxins',
    'Third generation cephalosporins resistance': 'Other',
    'Methicillin resistance': 'Other',
}

rows = []
for folder in [RES_DIR, RES_2020]:
    for fn in sorted(os.listdir(folder)):
        if not fn.endswith('.csv'):
            continue
        yr = get_year(fn)
        if yr is None:
            continue
        for r in extract_boxplots(folder / fn):
            r['Year'] = yr
            rows.append(r)

res = pd.DataFrame(rows)
for c in ['InterpretableAST', 'Resistant', 'ResistancePercentage']:
    res[c] = pd.to_numeric(res[c], errors='coerce')
res['Year'] = res['Year'].astype(int)
res = res.rename(columns={'Specimen': 'InfectionType'})
res['AntibioticClass'] = res['AntibioticName'].map(ABX_CLASS).fillna('Other')
res = res[res['Iso3'].isin(PAPER_COUNTRIES)]
print(f'resistance rows (44 countries, 2020-2023): {len(res):,}')

resistance rows (44 countries, 2020-2023): 8,250


## 4. Consumption (DID) — train-period imputation (fix D4)
Missing DID values are filled with the median of **observed 2020–2021 consumption**, so no
evaluation-period statistic enters any feature. The legacy file used a full-panel median.

In [4]:
did = pd.DataFrame(extract_did(DID_DIR) + extract_did(RES_2020))
did = did[did['Class'] == 'Antibiotics'][['Iso3', 'Year', 'DID']].rename(columns={'DID': 'Total_DID'})

train_did_median = did[did['Year'].isin([2020, 2021])]['Total_DID'].median()
print(f'train-period DID median: {train_did_median:.4f}  (legacy full-panel median: 18.8878)')

full = res.merge(did, on=['Iso3', 'Year'], how='left')
full['Total_DID'] = full['Total_DID'].fillna(train_did_median)

train-period DID median: 16.8425  (legacy full-panel median: 18.8878)


## 5. True same-series lags (fixes D1 + D2)
Group by (country, pathogen, antibiotic, **specimen**), sort by year, shift once. No
cross-specimen chaining and no self-lag. Verification below confirms the lag equals the true
prior with zero deviation wherever a prior exists.

In [5]:
full = full.sort_values(['Iso3', 'PathogenName', 'AntibioticName', 'InfectionType', 'Year'])
g = full.groupby(['Iso3', 'PathogenName', 'AntibioticName', 'InfectionType'], group_keys=False)
full['Resistance_lag1'] = g['ResistancePercentage'].shift(1)
full['DID_lag1']        = g['Total_DID'].shift(1)
full['Resistance_YoY_change'] = full['ResistancePercentage'] - full['Resistance_lag1']
full['Resistance_rolling2yr'] = (full['ResistancePercentage'] + full['Resistance_lag1']) / 2
print('same-series lags computed')

same-series lags computed


## 6. Time-causal surveillance flags (fix D3)
For a row in year *t*, the count includes only records from years strictly before *t*.
The legacy file used each country's total over all four years (correlation 0.99 with the
full panel), leaking future reporting volume into training rows.

In [6]:
yr_counts = res.groupby(['Iso3', 'Year']).size().unstack(fill_value=0)
cumulative_prior = yr_counts.cumsum(axis=1).shift(1, axis=1).fillna(0)
full['Country_Observation_Count'] = [
    cumulative_prior.loc[i, y] if y in cumulative_prior.columns else 0
    for i, y in zip(full['Iso3'], full['Year'])
]
full['High_Quality_Country'] = (full['Country_Observation_Count'] >= HQ_THRESHOLD).astype(int)
print('time-causal flags computed')

time-causal flags computed


## 7. Target years, lag imputation, income groups

In [7]:
df = full[full['Year'].isin(TARGET_YEARS)].copy()

# missing lags: pathogen-antibiotic median from the 2021 training year, then pathogen, then global
train_rows = df[df['Year'] == 2021]
n_missing_before = df['Resistance_lag1'].isna().sum()

pa_median = train_rows.groupby(['PathogenName', 'AntibioticName'])['Resistance_lag1'].median()
p_median  = train_rows.groupby('PathogenName')['Resistance_lag1'].median()
g_median  = train_rows['Resistance_lag1'].median()

pa_key = df['PathogenName'] + '||' + df['AntibioticName']
pa_map = {f'{p}||{a}': v for (p, a), v in pa_median.items()}
df['Resistance_lag1'] = df['Resistance_lag1'].fillna(pa_key.map(pa_map))
df['Resistance_lag1'] = df['Resistance_lag1'].fillna(df['PathogenName'].map(p_median))
df['Resistance_lag1'] = df['Resistance_lag1'].fillna(g_median)
df['DID_lag1'] = df['DID_lag1'].fillna(train_did_median)
df['Resistance_YoY_change'] = df['ResistancePercentage'] - df['Resistance_lag1']
df['Resistance_rolling2yr'] = (df['ResistancePercentage'] + df['Resistance_lag1']) / 2
print(f'lag1 missing before imputation: {n_missing_before} of {len(df)} ({100*n_missing_before/len(df):.1f}%)')

xl = pd.read_excel(XLSX, sheet_name='Country Analytical History', header=None)
year_row = xl.iloc[5].tolist()
col_idx = {int(year_row[c]): c for c in range(2, len(year_row))
           if pd.notna(year_row[c]) and str(year_row[c]).isdigit()}
INC = {'L': 'Low', 'LM': 'Lower-middle', 'UM': 'Upper-middle', 'H': 'High', '..': 'Unknown'}
inc = {}
for _, r in xl.iloc[11:].iterrows():
    iso = str(r[0])
    if len(iso) != 3 or not iso.isalpha():
        continue
    v = str(r[col_idx[2019]]).strip() if pd.notna(r[col_idx[2019]]) else '..'
    inc[iso] = INC.get(v, 'Unknown')
df['IncomeGroup'] = df['Iso3'].map(inc).fillna('Unknown')

lag1 missing before imputation: 589 of 6276 (9.4%)


## 8. Finalize, save, and verify

In [8]:
iso_map = res.drop_duplicates('Iso3').set_index('Iso3')[['CountryTerritoryArea', 'WHORegionName']]
df['CountryTerritoryArea'] = df['Iso3'].map(iso_map['CountryTerritoryArea'])
df['WHORegionName'] = df['Iso3'].map(iso_map['WHORegionName'])

cols = ['CountryTerritoryArea', 'Iso3', 'WHORegionName', 'Year', 'PathogenName',
        'AntibioticName', 'AntibioticClass', 'InfectionType',
        'ResistancePercentage', 'Resistant', 'InterpretableAST', 'Total_DID',
        'IncomeGroup', 'Country_Observation_Count', 'High_Quality_Country',
        'Resistance_lag1', 'DID_lag1', 'Resistance_YoY_change', 'Resistance_rolling2yr']
df = df[cols].reset_index(drop=True)
df.to_csv('AMR_Training_Dataset_Fixed.csv', index=False)

print(f'rows: {len(df):,}  countries: {df["CountryTerritoryArea"].nunique()}  '
      f'years: {sorted(df["Year"].unique())}')

# verification: lag equals the true same-series prior wherever one exists
key = ['CountryTerritoryArea', 'PathogenName', 'AntibioticName', 'InfectionType']
prev = df[df['Year'] == 2022][key + ['ResistancePercentage']].rename(
    columns={'ResistancePercentage': 'true_2022'})
t = df[df['Year'] == 2023].merge(prev, on=key, how='left')
have = t['true_2022'].notna()
dev = (t.loc[have, 'Resistance_lag1'] - t.loc[have, 'true_2022']).abs()
print(f'test rows with genuine prior: {have.sum()} of {len(t)} ({100*have.mean():.0f}%)')
print(f'max |lag - true prior|: {dev.max():.6f}   (legacy: 54% of matched rows off, median 8.6 pp)')

rows: 6,276  countries: 44  years: [np.int64(2021), np.int64(2022), np.int64(2023)]
test rows with genuine prior: 1928 of 2222 (87%)
max |lag - true prior|: 0.000000   (legacy: 54% of matched rows off, median 8.6 pp)
